##### Copyright 2026 Google LLC.

In [4]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Transcribe: Audio and Speech Recognition 🎙️

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_transcribe.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

This notebook will show you how to transcribe audio and speech into text using **Gemini Transcribe** models.

Whether you need to transcribe pre-recorded podcast files, clean up spoken filler words with smart formatting, generate synchronized subtitles with word-by-word timestamps, label different speakers in a meeting, or stream live audio from a microphone with low latency, Gemini makes it simple.

### What you will learn
1. **Transcribe pre-recorded audio**: Upload an audio file and get an accurate text transcript with automatic language detection.
2. **Steer language and spelling**: Guide transcription using standard language codes (like `es-ES` for Spanish).
3. **Custom vocabulary (speech biasing)**: Teach the model how to spell specialized words, brand names, or coffee shop items (`Cortado`, `Macchiato`).
4. **Smart transcription (disfluency removal & formatting)**: Automatically remove filler words ("um", "uh"), resolve self-corrections, and format spoken lists/numbers using `mode={"type": "smart"}`.
5. **Word-level timestamps**: Extract precise start and end times for every spoken word.
6. **Speaker identification (diarization)**: Automatically detect multiple speakers and format conversation turns.
7. **Programmatic subtitle export**: Build standard `.srt` subtitle caption files from timestamp annotations.
8. **Real-time live streaming**: Stream live audio chunks over WebSockets using `gemini-3.5-transcribe-live`.
9. **Secure client tokens**: Mint short-lived, restricted tokens for web and mobile apps.

> **What is the Interactions API?**
> The [Interactions API](https://ai.google.dev/gemini-api/docs/interactions) (`client.interactions.create`) is Gemini's unified interface for multimodal tasks, speech transcription, and agents. It handles file inputs, structured annotations (such as timestamps and speaker tags), and multi-turn workflows.

## Setup

### Install the SDK

Install the Google GenAI SDK (`google-genai` version 2.0 or higher) and `soundfile` for audio decoding:

In [5]:
%pip install -U -q "google-genai>=2.0.0" soundfile

### Set up your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see the [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) quickstart for a walkthrough.

In [6]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

### Select the transcription model

Set the model identifier for synchronous audio transcription:

In [7]:
MODEL_ID = "gemini-3.8-flash"  # @param ["gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash", "gemini-3.5-transcribe"] {"allow-input": true, "isTemplate": true}

print(f"Using transcription model: {MODEL_ID}")

Using transcription model: gemini-3.5-transcribe


## 1. Transcribe an audio file

To transcribe pre-recorded audio, you first upload the audio file using the [file API](./File_API.ipynb) (`client.files.upload`).

Then, pass the uploaded file to `client.interactions.create`. Gemini Transcribe automatically identifies the spoken language and returns the transcription:

In [8]:
import urllib.request
from IPython.display import Audio, display

# 1. Download a sample English audio file
audio_url = "https://storage.googleapis.com/generativeai-downloads/audio/tell-a-story.wav"
urllib.request.urlretrieve(audio_url, "tell-a-story.wav")

# 2. Listen to the sample audio
display(Audio(filename="tell-a-story.wav"))

# 3. Upload the audio file to the File API
audio_file = client.files.upload(file="tell-a-story.wav")

In [9]:
# 3. Request transcription using client.interactions.create
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "audio", "uri": audio_file.uri}],
)

print("Transcription result:")
print(interaction.output_text)

Transcription result:
Hello, tell me a short story about a rabbit. Start by whispering as if it's a secret, then get gradually louder.


## 2. Steer language with language codes

When you know what language is spoken in the audio, you can guide the model by passing standard language codes (such as `["es-ES"]` for Spanish or `["fr-FR"]` for French) in `transcription_config.language_codes`.

This ensures the model applies correct regional grammar, accents, and punctuation:

In [10]:
# Download a sample Spanish audio recording
urllib.request.urlretrieve(
    "https://storage.googleapis.com/cloud-samples-data/generative-ai/audio/spanish.wav",
    "spanish.wav",
)

display(Audio(filename="spanish.wav"))

# Upload the Spanish audio file
spanish_file = client.files.upload(file="spanish.wav")

# Transcribe with explicit Spanish language steering
interaction_es = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "audio", "uri": spanish_file.uri}],
    generation_config={
        "transcription_config": {
            "language_codes": ["es-ES"],  # Specify Spanish (Spain)
        }
    },
)

print("Spanish transcription result:")
print(interaction_es.output_text)

Spanish transcription result:
Hola, gracias por tu trabajo hoy. Espero que tengas un gran día.


## 3. Custom vocabulary (speech biasing)

Speech models can sometimes confuse rare technical terms, brand names, or specialized menu items with common words (e.g. hearing *"Cortado"* as *"caught auto"*).

To fix this, pass a list of domain terms in `custom_vocabulary`. This hints to the model to prioritize these specific words:

In [11]:
# Download an audio recording of a coffee order
urllib.request.urlretrieve(
    "https://storage.googleapis.com/cloud-samples-data/generative-ai/audio/coffee_order.wav",
    "coffee_order.wav",
)

display(Audio(filename="coffee_order.wav"))

# Upload the coffee order audio
coffee_file = client.files.upload(file="coffee_order.wav")

# First try without custom vocabulary
interaction_coffee = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "audio", "uri": coffee_file.uri}],
    generation_config={
        "transcription_config": {
            "language_codes": ["en-US"],
        }
    },
)

print("Transcription without custom vocabulary:")
print(interaction_coffee.output_text)

# Provide custom vocabulary hints for coffee terminology
interaction_coffee = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "audio", "uri": coffee_file.uri}],
    generation_config={
        "transcription_config": {
            "language_codes": ["en-US"],
            "custom_vocabulary": ["Cortado", "Macchiato", "oatmilk", "barista"],
        }
    },
)

print("Transcription with custom vocabulary:")
print(interaction_coffee.output_text)

Transcription without custom vocabulary:
Can I please have a 12 oz iced oat milk latte? Oh, and can I please have a hot matcha latte also with oat milk?
Transcription with custom vocabulary:
Can I please have a 12 oz iced oatmilk latte? Oh, and can I please have a hot matcha latte also with oatmilk?


## 4. Smart transcription (disfluency removal and formatting)

By default, Gemini Transcribe operates in **verbatim** mode (`mode={"type": "verbatim"}`), preserving every spoken word, filler sound (*"um"*, *"uh"*, *"like"*), repetition, and false start.

When transcribing meeting notes, dictations, or voice memos for human reading, enable **Smart transcription** by setting `mode={"type": "smart"}` in `transcription_config`.

### Key Benefits of Smart Transcription:
* **Disfluency removal**: Strips filler words (*"um"*, *"uh"*, *"you know"*), stuttering, and conversational hesitation.
* **Inline self-corrections**: Resolves verbal corrections directly (e.g. *"Let's meet Tuesday, actually wait, Wednesday"* becomes *"Let's meet Wednesday"*).
* **Automatic structured formatting**: Automatically formats bullet points, numbered lists, paragraphs, dates, currencies, and numbers.
* **Grammatical cleanup**: Applies natural capitalization and punctuation polish.

| Spoken audio | `verbatim` mode (Default) | `smart` mode (Smart transcription) |
| :--- | :--- | :--- |
| "Um, so for the meeting, I think we should, uh, invite Alice and, wait no, Bob and Carol." | "Um so for the meeting I think we should uh invite Alice and wait no Bob and Carol." | "For the meeting, I think we should invite Bob and Carol." |
| "First item review budget second item finalize timeline third item send recap" | "first item review budget second item finalize timeline third item send recap" | "1. Review budget<br>2. Finalize timeline<br>3. Send recap" |

> [!NOTE]
> Smart transcription (`"type": "smart"`) is optimized for clean reading. It is mutually exclusive with `timestamp_granularities` and `diarization_mode` (which require `{"type": "verbatim", ...}`).

In [18]:
# Download a conversational audio recording with filler words and self-corrections
audio_disfluency_url = (
    "https://storage.googleapis.com/generativeai-downloads/audio/rehearsing.wav"
)
urllib.request.urlretrieve(audio_disfluency_url, "rehearsing.wav")

display(Audio(filename="rehearsing.wav"))

# Upload the audio file
rehearsing_file = client.files.upload(file="rehearsing.wav")

# 1. Verbatim mode (Default: exact literal speech)
interaction_verbatim = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "audio", "uri": rehearsing_file.uri}],
    generation_config={
        "transcription_config": {
            "mode": {
                "type": "verbatim",
            },
        }
    },
)

print("--- Verbatim transcription (raw speech) ---")
print(interaction_verbatim.output_text)

# 2. Smart transcription mode (cleaned & formatted)
interaction_smart = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "audio", "uri": rehearsing_file.uri}],
    generation_config={
        "transcription_config": {
            "mode": {
                "type": "smart",
            },
        }
    },
)

print("\n--- Smart transcription (disfluencies removed & structured) ---")
print(interaction_smart.output_text)

--- Verbatim transcription (raw speech) ---
Uh, hello. Good evening, everyone. Um, I'd like to start by, well, first of all, thank you all for coming. Today is, um, a very special day, or rather, evening? No, afternoon? Right, evening. We are here to celebrate, uh, sorry, let me just find my notes. Ah, here. We are here to honor, no, not honor, but, um, to mark the launch of our new, sorry, my glasses are a bit foggy, the new marketing campaign. No, wait, product campaign? Product, yes. Um, where was I? Ah, yes. It has been a long journey, a very, uh, challenging, well, not challenging in a bad way, but, you know, difficult? No, rewarding. Rewarding is the word. So, um, yes, cheers to, wait, we don't have glasses yet. Thank you.

--- Smart transcription (disfluencies removed & structured) ---
Good evening everyone. First of all, thank you all for coming. Today is a very special evening. We are here to mark the launch of our new product campaign.

It has been a long journey, a very rewa

## 5. Understanding the response structure

Before extracting word timestamps and speaker labels, let's look at how Gemini formats detailed transcription responses.

When you request timestamps or speaker labels, the `interaction` object contains structured **annotations**:

```json
{
  "output_text": "Tell me a story...",
  "steps": [
    {
      "content": [
        {
          "text": "Tell me a story...",
          "annotations": [
            {
              "type": "word_info",
              "text": "Tell",
              "start_offset": "0.0s",
              "end_offset": "0.4s",
              "speaker": "spk:0"
            },
            {
              "type": "word_info",
              "text": "me",
              "start_offset": "0.4s",
              "end_offset": "0.7s",
              "speaker": "spk:0"
            }
          ]
        }
      ]
    }
  ]
}
```

With this mental model, navigating the response tree (`interaction.steps -> step.content -> annotations`) becomes straightforward!

### Extract word-level timestamps

Set `timestamp_granularities=["word"]` to get the exact start and end time for every word:

In [13]:
# Global list to store word timestamps for subtitle generation later
extracted_words = []

# Request word-level timestamps in the transcription config
interaction_words = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "audio", "uri": audio_file.uri}],
    generation_config={
        "transcription_config": {
            "timestamp_granularities": ["word"],
        }
    },
)

print("Extracted word timestamps:")
for step in interaction_words.steps:
    for item in step.content:
        if hasattr(item, "annotations") and item.annotations:
            for ann in item.annotations:
                if getattr(ann, "type", None) == "word_info":
                    extracted_words.append({
                        "text": ann.text,
                        "start": ann.start_offset,
                        "end": ann.end_offset,
                    })
                    print(f"[{ann.start_offset:>7} -> {ann.end_offset:>7}] {ann.text}")

Extracted word timestamps:
[ 0.400s ->  1.100s] Hello.
[ 1.600s ->  1.800s] Tell
[ 1.800s ->      2s] me
[     2s ->  2.100s] a
[ 2.100s ->  2.600s] short
[ 2.600s ->  3.200s] story
[ 3.200s ->  3.600s] about
[ 3.600s ->  3.700s] a
[ 3.700s ->  4.600s] rabbit.
[ 4.800s ->  5.100s] Start
[ 5.100s ->  5.400s] by
[ 5.400s ->  6.200s] whispering
[ 6.200s ->  6.400s] as
[ 6.400s ->  6.600s] if
[ 6.600s ->  6.800s] it's
[ 6.800s ->  6.900s] a
[ 6.900s ->  7.800s] secret,
[ 8.100s ->  8.300s] then
[ 8.300s ->  8.600s] get
[ 8.600s ->      9s] gradual


## 6. Speaker identification (diarization)

**Speaker diarization** is the process of identifying "who spoke when" in an audio recording.

Enable it by setting `diarization_mode="speaker"`. The model tags each word with a speaker identifier (e.g. `spk:0`, `spk:1`):

In [14]:
# Download an audio recording of an argument about pain au chocolats
urllib.request.urlretrieve(
    "https://storage.googleapis.com/generativeai-downloads/audio/pain_au_chocolat.wav",
    "pain_au_chocolat.wav",
)

display(Audio(filename="pain_au_chocolat.wav"))

# Upload the coffee order audio
pain_au_chocolat_file = client.files.upload(file="pain_au_chocolat.wav")

# Enable speaker diarization to separate distinct speakers
interaction_diarized = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "audio", "uri": pain_au_chocolat_file.uri}],
    generation_config={
        "transcription_config": {
            "diarization_mode": "speaker",
            "timestamp_granularities": ["word"],
        }
    },
)

# Group words into conversational turns by speaker
print("Diarized conversation turns:")
current_speaker = None
current_turn = []

for step in interaction_diarized.steps:
    for item in step.content:
        if hasattr(item, "annotations") and item.annotations:
            for ann in item.annotations:
                if getattr(ann, "type", None) == "word_info":
                    speaker = getattr(ann, "speaker", "spk:0")
                    # When the speaker changes, print the completed turn
                    if speaker != current_speaker:
                        if current_turn:
                            print(f"[{current_speaker}]: {' '.join(current_turn)}")
                        current_speaker = speaker
                        current_turn = [ann.text]
                    else:
                        current_turn.append(ann.text)

# Print the final speaker turn
if current_turn:
    print(f"[{current_speaker}]: {' '.join(current_turn)}")

Diarized conversation turns:
[spk:0]: One chocolatine, please.
[spk:1]: Tiago, arrête. It is a pain au chocolat.
[spk:0]: Wait, a guy from the south west told me it's chocolatine.
[spk:1]: Do not listen to them. 90% of France and the entire universe calls it pain au chocolat. Chocolatine is a miss.
[spk:0]: Meu Deus, you French are intense. In Brazil, people fight the exact same way over bolacha versus biscoito.
[spk:1]: Well, here pain au chocolat is the only real word.
[spk:0]: Fine. Two pain au chocolat, please. As long as it has chocolate, tá valendo.


## 7. Export subtitles (SRT format)

**SubRip (`.srt`)** is the standard subtitle text format used by video players (like YouTube, VLC, and video editors).

Here, you convert your extracted word timestamps (`extracted_words` from Step 5) programmatically into a valid `.srt` file:

In [15]:
def parse_seconds(time_str: str) -> float:
    """Parses offset string like '0.800s' or '1s' into float seconds."""
    return float(str(time_str).rstrip("s"))


def format_srt_time(seconds: float) -> str:
    """Converts float seconds into SRT timestamp format: HH:MM:SS,mmm."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int(round((seconds - int(seconds)) * 1000))
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"


# Programmatically generate subtitles from extracted_words
srt_filename = "subtitles.srt"
with open(srt_filename, "w", encoding="utf-8") as f:
    if extracted_words:
        start_sec = parse_seconds(extracted_words[0]["start"])
        end_sec = parse_seconds(extracted_words[-1]["end"])
        sentence_text = " ".join(w["text"] for w in extracted_words)

        f.write("1\n")
        f.write(f"{format_srt_time(start_sec)} --> {format_srt_time(end_sec)}\n")
        f.write(f"{sentence_text}\n\n")

print("--- Generated subtitles.srt ---")
with open(srt_filename, "r") as f:
    print(f.read())

--- Generated subtitles.srt ---
1
00:00:00,400 --> 00:00:09,000
Hello. Tell me a short story about a rabbit. Start by whispering as if it's a secret, then get gradual




## 8. Real-time live streaming transcription

Now that you have mastered pre-recorded audio transcription, let's explore **Real-Time Live Streaming**.

### The 4 Core Streaming Concepts Explained Simply
1. **Unary vs. Live Streaming**:
   * *Unary (File API)*: Like sending a recorded voice note. You upload the whole file and receive the full transcript after it finishes.
   * *Live Streaming (Live API)*: Like an active phone call. You open a continuous two-way connection (WebSocket) and receive text tokens immediately as words are spoken.
2. **What is PCM Audio?**: The Live API expects uncompressed raw audio bytes (**16 kHz mono 16-bit linear PCM**). Standard WAV audio files store PCM bytes directly without compression.
3. **Why 100ms Chunks?**: Slicing audio into small 100-millisecond slices lets the model transcribe speech with minimal delay.
4. **Why Async Python (`asyncio`)?**: Standard Python code runs line-by-line (blocking). Streaming audio requires **doing two things at once**:
   - Continuously sending audio chunks in the background using `session.send_realtime_input`.
   - Simultaneously listening for incoming real-time transcript text from the server.

### Quick Async Syntax Primer
| Keyword | What it means in plain English |
| :--- | :--- |
| `async def` | Defines a background task function that can pause while waiting for network data without freezing your app. |
| `await` | Pauses execution until a network request (like sending an audio frame) completes. |
| `asyncio.create_task()` | Starts a task running in the background immediately in parallel with other code. |
| `client.aio` | Stands for **A**synchronous **I**nput/**O**utput in the Google GenAI SDK. |

Gemini 3.5 Transcribe Live (`gemini-3.5-transcribe-live`) connects via `client.aio.live.connect`:

In [16]:
import asyncio
import wave
from google.genai import types

LIVE_MODEL_ID = "gemini-3.5-transcribe-live"  # @param ["gemini-3.5-transcribe-live"] {"allow-input": true, "isTemplate": true}


# Task 1: Background worker that reads audio and sends chunks
async def stream_audio_file(session, file_path: str, chunk_ms: int = 100):
    """Reads audio file in 100ms chunks and streams them to the Live API session."""
    with wave.open(file_path, "rb") as wf:
        rate = wf.getframerate()
        frames_per_chunk = int(rate * (chunk_ms / 1000.0))
        print(f"Streaming '{file_path}' ({rate}Hz, {frames_per_chunk} frames per chunk)...")

        while True:
            data = wf.readframes(frames_per_chunk)
            if not data:
                break
            # Send live binary audio chunk to the model
            await session.send_realtime_input(
                audio=types.Blob(data=data, mime_type=f"audio/pcm;rate={rate}")
            )
            # Yield execution briefly so the receiver task can process incoming tokens
            await asyncio.sleep(chunk_ms / 1000.0)

        # Signal that audio streaming has ended
        await session.send_realtime_input(audio_stream_end=True)
        print("Finished sending audio stream.")


# Task 2: Background worker that listens for incoming real-time text
async def receive_transcripts(session):
    """Receives real-time transcript text tokens from the model."""
    async for message in session.receive():
        server_content = getattr(message, "server_content", None)
        if not server_content:
            continue
        # Print incoming real-time transcript text
        if (
            getattr(server_content, "input_transcription", None)
            and server_content.input_transcription.text
        ):
            print(
                f"[Live Transcript]: {server_content.input_transcription.text}",
                flush=True,
            )
        if getattr(server_content, "turn_complete", False):
            print("\n[Turn Complete]", flush=True)


async def run_live_transcription(audio_path: str):
    """Connects to Gemini Live API and streams audio in real-time."""
    # Configure the real-time transcription session (SMART or VERBATIM mode)
    live_config = types.LiveConnectConfig(
        response_modalities=["TEXT"],
        input_audio_transcription=types.AudioTranscriptionConfig(
            mode="SMART",
            language_codes=["en-US"],
            custom_vocabulary=["Cortado", "Macchiato", "oatmilk"],
        ),
    )

    # Connect to the Live API WebSocket and stream audio
    async with client.aio.live.connect(model=LIVE_MODEL_ID, config=live_config) as session:
        # Step A: Launch receiver listening task in the background
        receive_task = asyncio.create_task(receive_transcripts(session))
        # Step B: Stream audio file chunks simultaneously
        await stream_audio_file(session, audio_path)
        # Step C: Wait briefly for final text responses, then close
        await asyncio.sleep(2)
        receive_task.cancel()


# Run live streaming transcription in the active notebook event loop
await run_live_transcription("tell-a-story.wav")

Streaming 'tell-a-story.wav' (24000Hz, 2400 frames per chunk)...
[Live Transcript]: Hello, tell me a short story about a rabbit. Start by whispering as if it's a secret.
Finished sending audio stream.
[Live Transcript]: Then get


## 9. Secure client tokens (ephemeral tokens)

When building web browser or mobile apps, **never hardcode your primary API key** in frontend client code.

Instead, use the standard **valet key pattern**:

```
[ Backend Server ] (Holds secret primary key)
        │
        ▼ (Mints short-lived 10-minute token restricted ONLY to Transcribe)
[ Frontend App / Webpage ] (Receives restricted ephemeral token)
        │
        ▼ (Connects securely to Gemini Live API without exposing your primary key!)
[ Gemini Transcribe Live API ]
```

### What are Token Constraints?
Constraints specify exactly which model (e.g. only `gemini-3.5-transcribe-live`) this temporary token is allowed to call. Even if the token were intercepted, it cannot be used for unauthorized text or image generation.

In [17]:
import datetime
from google.genai import types

# =========================================================
# Step 1: On your Backend Server (Python)
# =========================================================
# Mint a 10-minute token restricted strictly to Gemini Transcribe Live
expire_time = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(minutes=10)

token_response = client.auth_tokens.create(
    config=types.CreateAuthTokenConfig(
        uses=1,
        expire_time=expire_time,
        live_connect_constraints=types.LiveConnectConstraints(
            model=LIVE_MODEL_ID,
            config=types.LiveConnectConfig(
                response_modalities=["TEXT"],
                input_audio_transcription=types.AudioTranscriptionConfig(
                    language_codes=["en-US"],
                ),
            ),
        ),
    )
)

token_string = getattr(
    token_response,
    "name",
    getattr(token_response, "token", "auth_tokens/c19a_sample_token"),
)
print("Backend: Ephemeral token minted successfully:")
print(f"  Token:         {str(token_string)[:16]}... (truncated)")
print(f"  Expiry:        {expire_time.isoformat()}")
print(f"  Allowed model: {LIVE_MODEL_ID}")

# =========================================================
# Step 2: On your Frontend Client App (Web / Mobile / Device)
# =========================================================
# The frontend client initializes using ONLY the restricted token (no secret keys!):
frontend_client = genai.Client(api_key=token_string)
print("\nFrontend: Client initialized securely with restricted token!")

/tmp/ipykernel_1326/3091776122.py:10: ExperimentalWarning: The SDK's token creation implementation is experimental, and may change in future versions.
  token_response = client.auth_tokens.create(


Backend: Ephemeral token minted successfully:
  Token:         auth_tokens/a36d... (truncated)
  Expiry:        2026-08-26T17:05:16.862166+00:00
  Allowed model: gemini-3.5-transcribe-live

Frontend: Client initialized securely with restricted token!


## Next steps

Now that you understand speech recognition with Gemini Transcribe, explore these resources to build voice applications:

- **Smart Transcription**: Learn more about disfluency removal and formatting in the [Smart Transcription Guide](https://ai.google.dev/gemini-api/docs/transcribe#transcription-modes).
- **Audio Understanding**: Read the official [Gemini Audio Guide](https://ai.google.dev/gemini-api/docs/audio).
- **Live API Documentation**: Learn more about real-time streaming in the [Gemini Live API Overview](https://ai.google.dev/gemini-api/docs/live).
- **Ephemeral Tokens**: Review security best practices in the [Ephemeral Tokens Guide](https://ai.google.dev/gemini-api/docs/tokens).
- **Gemini API Cookbook**: Discover more tutorials in the [Gemini API Cookbook](https://github.com/google-gemini/cookbook).